In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [3]:
#loading the trained model, scaler pickle file, OHE for prediction
model=load_model('model.h5')

#load encoder and scaler
with open('label_encoder_gender.pk1','rb') as file:
    label_encoder_gender=pickle.load(file)
with open('ohe_geo.pk1','rb') as file:
    ohe_geo=pickle.load(file)
with open('scaler.pk1','rb') as file:
    scaler=pickle.load(file)

In [14]:
# Example input data
input_data = {
    'CreditScore': 450,
    'Geography': 'Germany',
    'Gender': 'Female',
    'Age': 50,
    'Tenure': 3,
    'Balance': 40000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 30000
}

In [15]:
# One-hot encode 'Geography'
geo_encoded = ohe_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=ohe_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\sowja\OneDrive\Desktop\ANN\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,0.0,1.0,0.0


In [16]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,450,Germany,Female,50,3,40000,2,1,1,30000


In [17]:
## Encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,450,Germany,0,50,3,40000,2,1,1,30000


In [18]:
## concatination one hot encoded 
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,450,0,50,3,40000,2,1,1,30000,0.0,1.0,0.0


In [19]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-2.09264482, -1.09499335,  1.05551796, -0.69539349, -0.57803098,
         0.80843615,  0.64920267,  0.97481699, -1.22456561, -0.99850112,
         1.72572313, -0.57638802]])

In [20]:
## PRedict churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 649ms/step


array([[0.3388734]], dtype=float32)

In [21]:
prediction_proba = prediction[0][0]

In [22]:
prediction_proba

np.float32(0.3388734)

In [23]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
